# Graph Neural Networks Expressive Power :

## Separability :


Encode the following graphs to match the input format of the GIN model you implemented in the previous session (remember to consider the adjacency matrix).
<img src="image/graph.png" width="600"></img>

Using the GIN model implemented during the previous practical session, analyze the inference results on the two graphs shown below. 
Can you provide an explanation for the observed outcomes?

## Counting power :
Execute the command bellow to load the dataset.

In [4]:
from preparecounting import ExpressiveDataset

In [5]:
import torch_geometric as tg
import numpy as np
import torch
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch.nn import Linear, Parameter
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops, degree
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINConv,global_mean_pool
from torch.nn import Sequential, Linear, ReLU

In [6]:
dataset = ExpressiveDataset('dataset/subgraphcount')
len(dataset)
dataset = dataset.shuffle()
train_dataset = dataset[:4000]
val_dataset = dataset[4000:4500]
test_dataset = dataset[4500:]


print('Number of training graphs: ',len(train_dataset))
print('Number of validation graphs: ',len(test_dataset))
print('Number of test graphs: ',len(test_dataset))

Number of training graphs:  4000
Number of validation graphs:  500
Number of test graphs:  500


In [7]:
class GIN(torch.nn.Module):
    def __init__(self):
        super().__init__()

        nn1 = Sequential(
            Linear(dataset.num_features, 32),
            ReLU(),
            Linear(32, 32)
        )
        self.conv1 = GINConv(nn1)

        nn2 = Sequential(
            Linear(32, 32),
            ReLU(),
            Linear(32, 32)
        )
        self.conv2 = GINConv(nn2)

        # Final binary output → 1 logit
        self.lin = Linear(32, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))

        # GRAPH embedding (one per graph)
        x = global_mean_pool(x, batch)

        # one logit per graph
        logits = self.lin(x)

        return logits.view(-1)  # shape: (batch_size,)
    


In [8]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for data in loader:
        optimizer.zero_grad()
        out = model(data)

        y = data.y.view(-1).float()
        loss = criterion(out, y)

        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs

    return total_loss / len(loader.dataset)
    

def evaluate(model, loader):
    model.eval()
    correct = 0

    with torch.no_grad():
        for data in loader:
            out = model(data)
            y = data.y.view(-1)

            pred = (out > 0).float()
            correct += (pred == y).sum().item()

    return correct / len(loader.dataset)


The task for this dataset is binary classification. To address this, you will utilize the GIN model implemented in the previous practical session. However, prior to model training, the dataset must be split into training, validation, and test sets.

In [9]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [10]:
#classification model
my_model = GIN()
optimizer=torch.optim.RMSprop(my_model.parameters(), lr=0.001)
criterion = torch.nn.BCEWithLogitsLoss()


In [11]:
for epoch in range(1, 100):
    loss = train(my_model, train_loader, optimizer, criterion)
    val_acc = evaluate(my_model, val_loader)
    test_acc = evaluate(my_model, test_loader)
    print(f'Epoch: {epoch:02d}, Loss: {loss:.4f}, Val Acc: {val_acc:.4f}, Test Acc: {test_acc:.4f}')

Epoch: 01, Loss: 0.4948, Val Acc: 0.7380, Test Acc: 0.7540
Epoch: 02, Loss: 0.4310, Val Acc: 0.7360, Test Acc: 0.7580
Epoch: 03, Loss: 0.4102, Val Acc: 0.7380, Test Acc: 0.7540
Epoch: 04, Loss: 0.4106, Val Acc: 0.7280, Test Acc: 0.7660
Epoch: 05, Loss: 0.4074, Val Acc: 0.7540, Test Acc: 0.7780
Epoch: 06, Loss: 0.4059, Val Acc: 0.7380, Test Acc: 0.7540
Epoch: 07, Loss: 0.4049, Val Acc: 0.7560, Test Acc: 0.7560
Epoch: 08, Loss: 0.4013, Val Acc: 0.7400, Test Acc: 0.7720
Epoch: 09, Loss: 0.4054, Val Acc: 0.7440, Test Acc: 0.7680
Epoch: 10, Loss: 0.3990, Val Acc: 0.7380, Test Acc: 0.7540
Epoch: 11, Loss: 0.3979, Val Acc: 0.7480, Test Acc: 0.7740
Epoch: 12, Loss: 0.3986, Val Acc: 0.7420, Test Acc: 0.7600
Epoch: 13, Loss: 0.4005, Val Acc: 0.7300, Test Acc: 0.7680
Epoch: 14, Loss: 0.3981, Val Acc: 0.7420, Test Acc: 0.7600
Epoch: 15, Loss: 0.3994, Val Acc: 0.7360, Test Acc: 0.7620
Epoch: 16, Loss: 0.3995, Val Acc: 0.7440, Test Acc: 0.7780
Epoch: 17, Loss: 0.3967, Val Acc: 0.7460, Test Acc: 0.75

In [12]:
test_acc

0.756

Generate the confusion matrix for the predictions made by your model.

In [23]:
from sklearn.metrics import confusion_matrix
import torch
def get_confusion_matrix_sklearn(model, loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for data in loader:
            logits = model(data)
            preds = (logits > 0).float().cpu().tolist()
            labels = data.y.view(-1).float().cpu().tolist()

            all_preds.extend(preds)
            all_labels.extend(labels)

    cm = confusion_matrix(all_labels, all_preds)
    return cm


In [24]:
cm = get_confusion_matrix_sklearn(my_model, test_loader)
print(cm)


[[337  40]
 [ 82  41]]


In [ ]:
Does GIN achieved good results on this task?


Now, using PyTorch or by designing your own layer based on the lecture on expressive power, develop a GNN capable of solving this task. You can modify either the input features or the model architecture.

Hint: Pay close attention to the title of this section for guidance.


In [ ]:
Your_Model = 

Can you explain why your model managed to resolve this task?